# CrewAI Text to SQL agent with langchain

In [ ]:
!pip -qqq install crewai-tools
!pip -qqq install pip --progress-bar off
!pip -qqq install langchain-core --progress-bar off
!pip -qqq install langchain-community --progress-bar off
!pip -qqq install langchain-groq --progress-bar off
!pip -qqq install langchain-experimental --progress-bar off

In [ ]:
pip install crewai

In [ ]:
pip install langchain_huggingface

In [ ]:
pip install paramiko --upgrade

In [ ]:
%pip install --upgrade --quiet huggingface_hub

In [ ]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from textwrap import dedent
from typing import Any, Dict, List, Tuple, Union

from crewai import Agent, Crew, Process, Task
from crewai_tools import tool
from langchain.schema import AgentFinish
from langchain.schema.output import LLMResult
from langchain_community.tools.sql_database.tool import (
    InfoSQLDatabaseTool,
    ListSQLDatabaseTool,
    QuerySQLCheckerTool,
    QuerySQLDataBaseTool,
)
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_core.callbacks.base import BaseCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

In [ ]:
import os
from getpass import getpass

HUGGINGFACEHUB_API_TOKEN = getpass()
os.environ['HUGGINGFACEHUB_API_TOKEN'] = HUGGINGFACEHUB_API_TOKEN

In [ ]:
import sqlite3

# Create the SQLite database and table
conn = sqlite3.connect('dummy_data.db')
cursor = conn.cursor()

# Drop the sales table if it already exists
cursor.execute('DROP TABLE IF EXISTS sales')

# Create the sales table
cursor.execute('''
    CREATE TABLE sales (
        sl_no INTEGER PRIMARY KEY,
        product TEXT,
        shipping_location TEXT,
        shipping_costs INTEGER,
        actual_costs INTEGER,
        selling_price INTEGER,
        selling_location TEXT,
        profits INTEGER
    )
''')

# Insert dummy data into the sales table
data = [
    (1, 'Flask', 'Bangalore', 50, 200, 300, 'Mumbai', 100),
    (2, 'Flask', 'Delhi', 40, 200, 310, 'Pune', 110),
    (3, 'Flask', 'Chennai', 60, 200, 320, 'Hyderabad', 120),
    (6, 'Bottle', 'Bangalore', 30, 100, 200, 'Mumbai', 100),
    (7, 'Bottle', 'Delhi', 25, 100, 210, 'Pune', 110),
    (8, 'Bottle', 'Chennai', 35, 100, 220, 'Hyderabad', 120),
    (11, 'Mug', 'Bangalore', 20, 50, 100, 'Mumbai', 50),
    (12, 'Mug', 'Delhi', 18, 50, 110, 'Pune', 60),
    (13, 'Mug', 'Chennai', 22, 50, 120, 'Hyderabad', 70),
    (16, 'Cup', 'Bangalore', 15, 30, 70, 'Mumbai', 40),
    (17, 'Cup', 'Delhi', 12, 30, 75, 'Pune', 45),
    (18, 'Cup', 'Chennai', 18, 30, 80, 'Hyderabad', 50),
    (21, 'Plate', 'Bangalore', 25, 60, 120, 'Mumbai', 60),
    (22, 'Plate', 'Delhi', 22, 60, 125, 'Pune', 65),
    (23, 'Plate', 'Chennai', 28, 60, 130, 'Hyderabad', 70),
    (26, 'Pan', 'Bangalore', 35, 150, 250, 'Mumbai', 100),
    (27, 'Pan', 'Delhi', 32, 150, 260, 'Pune', 110),
    (28, 'Pan', 'Chennai', 38, 150, 270, 'Hyderabad', 120),
    (31, 'Pot', 'Bangalore', 25, 80, 160, 'Mumbai', 80),
    (32, 'Pot', 'Delhi', 22, 80, 165, 'Pune', 85),
    (33, 'Pot', 'Chennai', 28, 80, 170, 'Hyderabad', 90),
    (36, 'Bowl', 'Bangalore', 10, 20, 50, 'Mumbai', 30),
    (37, 'Bowl', 'Delhi', 8, 20, 55, 'Pune', 35),
    (38, 'Bowl', 'Chennai', 12, 20, 60, 'Hyderabad', 40),
    (41, 'Spoon', 'Bangalore', 5, 10, 30, 'Mumbai', 20),
    (42, 'Spoon', 'Delhi', 4, 10, 32, 'Pune', 22),
    (43, 'Spoon', 'Chennai', 6, 10, 34, 'Hyderabad', 24),
    (46, 'Fork', 'Bangalore', 5, 15, 35, 'Mumbai', 20),
    (47, 'Fork', 'Delhi', 4, 15, 38, 'Pune', 23),
    (48, 'Fork', 'Chennai', 6, 15, 41, 'Hyderabad', 26)
]

cursor.executemany('''
    INSERT INTO sales (sl_no, product, shipping_location, shipping_costs, actual_costs, selling_price, selling_location, profits)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
''', data)

conn.commit()
conn.close()

In [ ]:
from langchain.agents.agent_toolkits import SQLDatabaseToolkit 
from langchain.sql_database import SQLDatabase 

db = SQLDatabase.from_uri("sqlite:///dummy_data.db")
print("Database dialect:", db.dialect)
print("Usable table names:", db.get_usable_table_names())

# Run a test query
test_query_result = db.run("SELECT * FROM sales LIMIT 10;")
print("Test query result:", test_query_result)
print(db.get_table_info())

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint

model_id = "openai-community/gpt2-large"

llm = HuggingFaceEndpoint(
    repo_id=model_id,
    max_new_tokens=70,
    top_k=5,
    top_p=0.95,
    typical_p=0.95,
    temperature=0.2,
    stop_sequences = ["\n\n"],
    # repetition_penalty=1.03,
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN
)

# print(llm.invoke("What is Deep Learning?"))

In [ ]:
@tool("list_tables")
def list_tables() -> str:
    """List the available tables in the database"""
    return ListSQLDatabaseTool(db=db).invoke("")
     
list_tables.run()

In [ ]:
@tool("tables_schema")
def tables_schema(tables: str) -> str:
    """
    Input is a comma-separated list of tables, output is the schema and sample rows
    for those tables. Be sure that the tables actually exist by calling `list_tables` first!
    Example Input: table1, table2, table3
    """
    tool = InfoSQLDatabaseTool(db=db)
    return tool.invoke(tables)

print(tables_schema.run("sales"))

In [ ]:
@tool("execute_sql")
def execute_sql(sql_query: str) -> str:
    """Execute a SQL query against the database. Returns the result"""
    return QuerySQLDataBaseTool(db=db).invoke(sql_query)

execute_sql.run("SELECT * FROM sales WHERE actual_costs > 10 LIMIT 5")

In [ ]:
@tool("check_sql")
def check_sql(sql_query: str) -> str:
    """
    Use this tool to double check if your query is correct before executing it. Always use this
    tool before executing a query with `execute_sql`.
    """
    return QuerySQLCheckerTool(db=db, llm=llm).invoke({"query": sql_query})
     

check_sql.run("SELECT * WHERE actual_costs > 10000 LIMIT 5 table = sales")

In [ ]:
sql_dev = Agent(
    role="Senior Database Developer",
    goal="Construct and execute SQL queries based on a request",
    backstory=dedent(
        """
        You are an experienced database engineer who is master at creating efficient and complex SQL queries.
        You have a deep understanding of how different databases work and how to optimize queries.
        Use the `list_tables` to find available tables.
        Use the `tables_schema` to understand the metadata for the tables.
        Use the `execute_sql` to check your queries for correctness.
        Use the `check_sql` to execute queries against the database.
    """
    ),
    llm=llm,
    tools=[list_tables, tables_schema, execute_sql, check_sql],
    allow_delegation=False,
)

In [ ]:
data_analyst = Agent(
    role="Senior Data Analyst",
    goal="You receive data from the database developer and analyze it",
    backstory=dedent(
        """
        You have deep experience with analyzing datasets using Python.
        Your work is always based on the provided data and is clear,
        easy-to-understand and to the point. You have attention
        to detail and always produce very detailed work (as long as you need).
    """
    ),
    llm=llm,
    allow_delegation=False,
)

In [ ]:
report_writer = Agent(
    role="Senior Report Editor",
    goal="Write an executive summary type of report based on the work of the analyst",
    backstory=dedent(
        """
        Your writing still is well known for clear and effective communication.
        You always summarize long texts into bullet points that contain the most
        important details.
        """
    ),
    llm=llm,
    allow_delegation=False,
)

In [ ]:
extract_data = Task(
    description="Extract data that is required for the query {query}.",
    expected_output="Database result for the query",
    agent=sql_dev,
)

analyze_data = Task(
    description="Analyze the data from the database and write an analysis for {query}.",
    expected_output="Detailed analysis text",
    agent=data_analyst,
    context=[extract_data],
)   

write_report = Task(
    description=dedent(
        """
        Write an executive summary of the report from the analysis. The report
        must be less than 100 words.
    """
    ),
    expected_output="Markdown report",
    agent=report_writer,
    context=[analyze_data],
)

In [ ]:
crew = Crew(
    agents=[sql_dev, data_analyst, report_writer],
    tasks=[extract_data, analyze_data, write_report],
    process=Process.sequential,
    verbose=2,
    memory=False,
    output_log_file="crew.log",
)

inputs = {
    "query": "Total profits for each product type"
}

result = crew.kickoff(inputs=inputs)

In [ ]:
print(result)

In [ ]:
inputs = {
    "query": "Which location has sold the most pans"
}

result = crew.kickoff(inputs=inputs)

In [ ]:
print(result)